In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")
from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import get_telecommute

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

In [3]:
data_fullsurvey['Person'] = data_fullsurvey['Person'].merge(data_fullsurvey['Household'][['hhno', 'hhparcel', 'hhtaz']], on='hhno', how='left')
data_fullsurvey['Person'] = data_fullsurvey['Person'].merge(taz_subarea[['TAZ', 'County', 'Jurisdiction', 'DistrictFlowName', 'Subarea']], 
                                                    left_on='hhtaz', right_on='TAZ', how='left')

In [4]:
data_daysim['Person'] = data_daysim['Person'].merge(data_daysim['Household'][['hhno', 'hhparcel', 'hhtaz']], on='hhno', how='left')
data_daysim['Person'] = data_daysim['Person'].merge(taz_subarea[['TAZ', 'County', 'Jurisdiction', 'DistrictFlowName', 'Subarea']], 
                                                    left_on='hhtaz', right_on='TAZ', how='left')
data_daysim['PersonDay'] = data_daysim['PersonDay'].merge(data_daysim['Person'][['hhno', 'pno', 'pwtyp', 'pwpcl', 'hhparcel', 'hhtaz', 
                                                            'County', 'Jurisdiction', 'Subarea']], 
                                                            on=['hhno', 'pno'], how='left')

In [5]:
def show_wfh_table(df_survey, df_daysim):
    workers_survey = df_survey[['worker_type', 'psexpfac']].groupby('worker_type').sum().reset_index()
    workers_daysim = df_daysim[['worker_type', 'pdexpfac']].groupby('worker_type').sum().reset_index()
    workers_survey.columns = ['Telecommute Type', 'Number of Workers']
    workers_daysim.columns = ['Telecommute Type', 'Number of Workers']
    workers = workers_survey.merge(workers_daysim, on='Telecommute Type', how='left', suffixes=(' (Survey)', ' (Model)'))
    # show numbers
    workers = workers.set_index('Telecommute Type')
    workers = workers.loc[['Commuter', 'Telecommuter', 'WFH', 'Not Worker'], :]
    display(workers.style.format({'Number of Workers (Survey)': '{:,.0f}',
                                  'Number of Workers (Model)': '{:,.0f}'}))

def show_worker_table(df_survey, df_daysim):
    workers_survey = df_survey[['pwtyp', 'psexpfac']].groupby('pwtyp').sum().reset_index()
    workers_daysim = df_daysim[['pwtyp', 'psexpfac']].groupby('pwtyp').sum().reset_index()
    workers_survey.columns = ['Worker Type', 'Number of Workers']
    workers_daysim.columns = ['Worker Type', 'Number of Workers']
    workers = workers_survey.merge(workers_daysim, on='Worker Type', how='left', suffixes=(' (Survey)', ' (Model)'))
    # show numbers
    workers = workers.set_index('Worker Type')
    workers = workers.loc[['Paid Full-Time Worker', 'Paid Part-Time Worker', 'Not a Paid Worker'], :]
    display(workers.style.format({'Number of Workers (Survey)': '{:,.0f}',
                                  'Number of Workers (Model)': '{:,.0f}'}))

In [6]:
data_daysim['PersonDay'] = get_telecommute(data_daysim['PersonDay'])
data_fullsurvey['Person'] = get_telecommute(data_fullsurvey['Person'])

## PSRC Region

In [7]:
show_worker_table(data_fullsurvey['Person'], data_daysim['Person'])

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"1,784,635","1,873,612"
Paid Part-Time Worker,"487,739","538,846"
Not a Paid Worker,"1,948,808","1,893,697"


In [8]:
show_wfh_table(df_survey=data_fullsurvey['Person'], df_daysim=data_daysim['PersonDay'])

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"1,729,266","1,870,704"
Telecommuter,"258,530","378,479"
WFH,"284,578","163,275"
Not Worker,"1,948,808","1,893,697"


## King County

In [9]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']=='King']
df_daysim = data_daysim['Person'][data_daysim['Person']['County']=='King']
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"1,041,177","1,028,264"
Paid Part-Time Worker,"239,881","300,875"
Not a Paid Worker,"965,713","941,336"


In [10]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']=='King']
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['County']=='King']
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"912,632","969,524"
Telecommuter,"205,705","286,958"
WFH,"162,721","72,657"
Not Worker,"965,713","941,336"


## Out of King County

In [11]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']!='King']
df_daysim = data_daysim['Person'][data_daysim['Person']['County']!='King']
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"743,458","845,348"
Paid Part-Time Worker,"247,858","237,971"
Not a Paid Worker,"983,095","952,361"


In [12]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['County']!='King']
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['County']!='King']
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"816,634","901,180"
Telecommuter,"52,825","91,521"
WFH,"121,857","90,618"
Not Worker,"983,095","952,361"


## BKR Area

In [13]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"150,141","143,379"
Paid Part-Time Worker,"36,945","40,699"
Not a Paid Worker,"149,936","139,497"


In [14]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"127,404","127,410"
Telecommuter,"31,732","45,647"
WFH,"27,950","11,021"
Not Worker,"149,936","139,497"


## Out of BKR Area

In [15]:
df_survey = data_fullsurvey['Person'][~data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['Person'][~data_daysim['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"1,634,494","1,730,233"
Paid Part-Time Worker,"450,794","498,147"
Not a Paid Worker,"1,798,873","1,754,200"


In [16]:
df_survey = data_fullsurvey['Person'][~data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
df_daysim = data_daysim['PersonDay'][~data_daysim['PersonDay']['Jurisdiction'].isin(['BELLEVUE', 'KIRKLAND', 'REDMOND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"1,601,862","1,743,294"
Telecommuter,"226,798","332,832"
WFH,"256,628","152,254"
Not Worker,"1,798,873","1,754,200"


## City of Bellevue

In [17]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['BELLEVUE'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"70,392","65,337"
Paid Part-Time Worker,"21,394","19,362"
Not a Paid Worker,"66,567","70,357"


In [18]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['BELLEVUE'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['BELLEVUE'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"64,662","58,741"
Telecommuter,"14,133","20,865"
WFH,"12,991","5,093"
Not Worker,"66,567","70,357"


## City of Kirkland

In [19]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['KIRKLAND'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['KIRKLAND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"32,741","44,635"
Paid Part-Time Worker,"4,695","12,207"
Not a Paid Worker,"37,989","38,459"


In [20]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['KIRKLAND'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['KIRKLAND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"22,591","39,342"
Telecommuter,"11,043","13,967"
WFH,"3,802","3,533"
Not Worker,"37,989","38,459"


## City of Redmond

In [21]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['KIRKLAND'])]
df_daysim = data_daysim['Person'][data_daysim['Person']['Jurisdiction'].isin(['KIRKLAND'])]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"32,741","44,635"
Paid Part-Time Worker,"4,695","12,207"
Not a Paid Worker,"37,989","38,459"


In [22]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Jurisdiction'].isin(['REDMOND'])]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Jurisdiction'].isin(['REDMOND'])]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"40,151","29,327"
Telecommuter,"6,556","10,815"
WFH,"11,157","2,395"
Not Worker,"45,380","30,681"


## Bellevue Downtown

In [23]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Subarea']==3]
df_daysim = data_daysim['Person'][data_daysim['Person']['Subarea']==3]
show_worker_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Worker Type,,
Paid Full-Time Worker,"6,933","8,438"
Paid Part-Time Worker,343,"2,394"
Not a Paid Worker,"2,167","7,275"


In [24]:
df_survey = data_fullsurvey['Person'][data_fullsurvey['Person']['Subarea']==3]
df_daysim = data_daysim['PersonDay'][data_daysim['PersonDay']['Subarea']==3]
show_wfh_table(df_survey=df_survey, df_daysim=df_daysim)

,Number of Workers (Survey),Number of Workers (Model)
Telecommute Type,,
Commuter,"3,168","7,438"
Telecommuter,"2,003","2,643"
WFH,"2,106",751
Not Worker,"2,167","7,275"
